In [10]:
#0.모듈 사용
import tensorflow as tf
import pandas as pd
import os

In [11]:
#1. 데이터셋 준비
dataset_path = "CNN_train_test/CNN_train_test_image"

image_size = (224, 224)
batch_size = 32

# 트레이닝용 데이터셋
train_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=image_size,
    batch_size=batch_size
)

# 테스트용 데이터셋
test_dataset = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=image_size,
    batch_size=batch_size
)

# 클래스(무드) 이름 확인
class_names = train_dataset.class_names
print("인식된 패션 무드 목록:", class_names)

Found 120975 files belonging to 24 classes.
Using 96780 files for training.
Found 120975 files belonging to 24 classes.
Using 24195 files for validation.
인식된 패션 무드 목록: ['기타', '레트로', '로맨틱', '리조트', '매니시', '모던', '밀리터리', '섹시', '소피스트케이티드', '스트리트', '스포티', '아방가르드', '오리엔탈', '웨스턴', '젠더리스', '컨트리', '클래식', '키치', '톰보이', '펑크', '페미닌', '프레피', '히피', '힙합']


In [12]:
# 성능 최적화를 위해 prefetch 설정 [다음에 학습할 데이터를 미리 메모리에 로딩해놓기]
AUTOTUNE = tf.data.AUTOTUNE

train_ds= train_dataset.prefetch(buffer_size=AUTOTUNE)
test_ds= test_dataset.prefetch(buffer_size=AUTOTUNE)

In [13]:
# 전이학습에 필요한 사전학습모델 준비하기
mood_model = tf.keras.applications.convnext.ConvNeXtTiny(
    input_shape=(224, 224, 3),
    include_top=False,
    include_preprocessing=True,
    weights='imagenet',  # 👈 'imagnet'을 'imagenet'으로 올바르게 수정!
    pooling='avg'
)

mood_model.trainable = False

In [14]:
# 분류기 만들기 -- 입출력층
num_classes = len(class_names)  # 또는 직접 숫자 입력 (예: 3)

# 최종 분류 모델 구성
model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=(224, 224, 3)),
    mood_model,  # 특징 추출 (여기서 이미 1차원 벡터로 나옴)
    tf.keras.layers.Dense(256, activation='relu'), # 필요시 중간 Dense 추가 가능
    tf.keras.layers.Dense(len(class_names), activation='softmax') # 최종 무드 분류 출력층
])

In [15]:
#모델 체인 형태 만들기
inputs = tf.keras.layers.Input(shape=(224, 224, 3))
x = mood_model(inputs)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)
mood_classifier_model = tf.keras.Model(inputs=inputs, outputs=outputs)


# 모델 컴파일
mood_classifier_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

mood_classifier_model.summary()

Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_13 (InputLayer)     │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ convnext_tiny (Functional)      │ (None, 768)            │    27,820,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 24)             │        18,456 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,838,584 (106.20 MB)

 Trainable params: 18,456 (72.09 KB)

 Non-trainable params: 27,820,128 (106.13 MB)

In [17]:
#모델 학습
history= mood_classifier_model.fit(
    train_ds,
    validation_data=test_ds,
    batch_size=16,
    epochs=1
)

3025/3025 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4973 - loss: 1.6834

KeyboardInterrupt: 

In [ ]:
# 모델 저장
os.makedirs("Model", exist_ok=True)
model_path = "Model/fashion_mood_model_tensorflow.pth"
model.save(mood_model.state_dict(), model_path)

print(f"모델 성공적 저장: {model_path}")

In [ ]:
%pip install matplotlib

In [ ]:
# 학습결과 시각화
import matplotlib.pyplot as plt  # .pyplot을 붙여서 임포트해야 합니다!

plt.plot(history.history['accuracy'], label='train acc')
plt.plot(history.history['val_accuracy'], label='test acc')
plt.title('accuracy per epoch')
plt.xlabel('epochs')
plt.ylabel('accuracy')
plt.grid(True)
plt.legend()
plt.show()  

ModuleNotFoundError: No module named 'matplotlib'